In [7]:
import pandas as pd
import plotly.express as px


# Dataset: Gapminder — GDP, Life Expectancy, Population by Country
# Source: Gapminder Foundation (gapminder.org)

df = px.data.gapminder()
print(f"Loaded: {len(df)} rows")
print(df.head())


# explore

print(df.info())
print("Years:", sorted(df['year'].unique()))
print("Continents:", df['continent'].unique())
print(df.describe().round(1))


Loaded: 1704 rows
       country continent  year  lifeExp       pop   gdpPercap iso_alpha  \
0  Afghanistan      Asia  1952   28.801   8425333  779.445314       AFG   
1  Afghanistan      Asia  1957   30.332   9240934  820.853030       AFG   
2  Afghanistan      Asia  1962   31.997  10267083  853.100710       AFG   
3  Afghanistan      Asia  1967   34.020  11537966  836.197138       AFG   
4  Afghanistan      Asia  1972   36.088  13079460  739.981106       AFG   

   iso_num  
0        4  
1        4  
2        4  
3        4  
4        4  
<class 'pandas.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   str    
 1   continent  1704 non-null   str    
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
 6   iso_alpha  1704 non-null   str   

## Task 1 — Scatter: life expectancy change over time

**What to build:** A scatter showing **GDP per capita vs life expectancy** for **two years** (2002 and 2007) to show how both moved — use **colour for year** (just 2 colours), **one continent only**.

Choose any continent except Africa (that was the example). Highlight the change direction.

> 💡 Filter: `df.loc[df['continent'] == 'YOUR_CHOICE']` then filter years


In [8]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# 1. Load dataset
df = px.data.gapminder()

# 2. Filter for Europe and the years 2002 & 2007
continent_choice = 'Europe'
df_filtered = df[(df['continent'] == continent_choice) & (df['year'].isin([2002, 2007]))].copy()
df_filtered['year'] = df_filtered['year'].astype(str)

# 3. Create base scatter plot
fig = px.scatter(
    df_filtered,
    x='gdpPercap',
    y='lifeExp',
    color='year',
    text='country',
    log_x=True,
    color_discrete_sequence=['#3b82f6', '#ef4444'], # Distinct colours for 2002 and 2007
    title=f'GDP per capita vs Life Expectancy (2002 vs 2007) — {continent_choice}',
    labels={
        'gdpPercap': 'GDP per Capita (USD, Log Scale)',
        'lifeExp': 'Life Expectancy (Years)',
        'year': 'Year'
    }
)

# 4. Highlight change direction with connecting trajectory lines
for country in df_filtered['country'].unique():
    c_df = df_filtered[df_filtered['country'] == country].sort_values('year')
    if len(c_df) == 2:
        fig.add_trace(
            go.Scatter(
                x=c_df['gdpPercap'],
                y=c_df['lifeExp'],
                mode='lines',
                line=dict(color='rgba(150, 150, 150, 0.5)', width=1.5, dash='dot'),
                showlegend=False,
                hoverinfo='skip'
            )
        )

# 5. Styling and layout updates
fig.update_traces(textposition='top center', marker=dict(size=9))
fig.update_layout(
    template='plotly_white',
    width=900,
    height=650,
    font=dict(family='Inter, sans-serif', size=12)
)

fig.show()

## Task 2 — Bubble chart: tell a story

**What to build:** A bubble chart (full 2007 dataset, all countries) where:
- x = GDP per capita (log scale)
- y = life expectancy
- size = population
- colour = ONE continent highlighted (your choice), all others grey
- At least one annotation explaining the highlighted group's story

> This is the grey-and-highlight technique applied to a bubble chart.


The script below filters the 2007 Gapminder dataset, scales bubble sizes by country population, grays out all other continents while emphasizing Europe in a distinct blue, and adds an explanatory annotation highlighting the group's narrative.

In [9]:
import plotly.express as px
import plotly.graph_objects as go

# 1. Load the 2007 Gapminder dataset
df = px.data.gapminder().query("year == 2007")

# 2. Define custom color map to highlight ONE continent (Europe) and gray out the rest
color_map = {
    'Europe': '#3b82f6',     # Vibrant Blue for the highlighted group
    'Africa': '#e2e8f0',     # Neutral Light Gray
    'Americas': '#e2e8f0',
    'Asia': '#e2e8f0',
    'Oceania': '#e2e8f0'
}

# 3. Create the bubble chart
fig = px.scatter(
    df,
    x="gdpPercap",
    y="lifeExp",
    size="pop",
    color="continent",
    hover_name="country",
    log_x=True,
    size_max=55,
    color_discrete_map=color_map,
    title="<b>Global Health and Wealth in 2007: The European Cluster</b>",
    labels={
        "gdpPercap": "GDP per Capita (USD, Log Scale)",
        "lifeExp": "Life Expectancy (Years)",
        "pop": "Population",
        "continent": "Continent"
    }
)

# 4. Customize layout aesthetics
fig.update_layout(
    template="plotly_white",
    width=950,
    height=650,
    font=dict(family="Inter, sans-serif", size=12),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

# 5. Add strategic annotation explaining Europe's story
fig.add_annotation(
    x=35000,  # Data coordinate on log scale axis
    y=81.5,
    text="<b>The European Longevity Cluster</b><br>European nations exhibit high life expectancies (75–82 yrs)<br>coupled with high GDP per capita, forming a distinct compact tier.",
    showarrow=True,
    arrowhead=2,
    arrowsize=1,
    arrowwidth=2,
    arrowcolor="#3b82f6",
    ax=-140,
    ay=-60,
    bgcolor="rgba(255, 255, 255, 0.9)",
    bordercolor="#3b82f6",
    borderwidth=1,
    borderpad=6
)

# 6. Render the plot
fig.show()